# Cell Counting Evaluation

Evaluate cell counting performance across multiple BBBC datasets using five segmentation models.

**Datasets:** BBBC001, BBBC039, BBBC041

**Models:** cellpose4, cellpose3, microatlas, microsam, cellsam

**Metrics:**
- MAE — Mean Absolute Error
- RMSE — Root Mean Squared Error
- R² — Coefficient of Determination
- Pearson r — Pearson Correlation Coefficient
- MPE — Mean Percentage Error

In [ ]:
import sys
import os
import csv
import time
import numpy as np
from pathlib import Path

# Ensure counting directory is on the import path
SRC_DIR = Path.cwd().parent / 'src'
COUNTING_DIR = SRC_DIR / 'counting'
if str(COUNTING_DIR) not in sys.path:
    sys.path.insert(0, str(COUNTING_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import counting_utils as cu

## Dataset Configuration

In [ ]:
BBBC_ROOT = COUNTING_DIR / 'BBBC'

DATASET_CONFIGS = {
    'BBBC001': {
        'image_dir': str(BBBC_ROOT / 'BBBC001' / 'BBBC001_v1_images_tif' / 'human_ht29_colon_cancer_1_images'),
        'counts_file': str(BBBC_ROOT / 'BBBC001' / 'BBBC001_v1_counts.txt'),
        'n_images': 6,
    },
    'BBBC039': {
        'image_dir': str(BBBC_ROOT / 'BBBC039' / 'images'),
        'counts_file': str(BBBC_ROOT / 'BBBC039' / 'BBBC039_v1_counts.txt'),
        'n_images': 200,
    },
    'BBBC041': {
        'image_dir': str(BBBC_ROOT / 'BBBC041' / 'malaria' / 'images'),
        'counts_file': str(BBBC_ROOT / 'BBBC041' / 'BBBC041_v1_counts.txt'),
        'n_images': 1328,
    },
}

# List available datasets
for name, cfg in DATASET_CONFIGS.items():
    exists = os.path.isdir(cfg['image_dir'])
    status = 'OK' if exists else 'NOT FOUND'
    print(f'  {name}: {cfg["n_images"]:>5d} images  [{status}]')

## Configuration

Select which models and datasets to evaluate:

In [ ]:
# Select models and datasets to run
MODELS = ['microatlas']  # Options: 'cellpose4', 'cellpose3', 'microatlas', 'microsam', 'cellsam'
DATASETS = ['BBBC001', 'BBBC039', 'BBBC041']  # Options: 'BBBC001', 'BBBC039', 'BBBC041'
OUTPUT_DIR = str(COUNTING_DIR / 'results')
USE_GPU = True

print(f'Models: {MODELS}')
print(f'Datasets: {DATASETS}')
print(f'Output: {OUTPUT_DIR}')

## Run Counting Evaluation

In [ ]:
def run_single_task(model, dataset_name, output_root, use_gpu=True):
    """Execute a single (model, dataset) counting evaluation task."""
    cfg = DATASET_CONFIGS[dataset_name]
    image_dir = cfg['image_dir']
    counts_file = cfg['counts_file']
    output_dir = str(Path(output_root) / f'{dataset_name}_{model}_counting')

    if not os.path.isdir(image_dir):
        print(f'  [Skip] Image directory not found: {image_dir}')
        return False, 0, {}
    if not os.path.isfile(counts_file):
        print(f'  [Skip] Counts file not found: {counts_file}')
        return False, 0, {}

    os.makedirs(output_dir, exist_ok=True)
    start_t = time.time()

    try:
        if model == 'cellpose4':
            import eval_cellpose4_counting
            result_data = eval_cellpose4_counting.run_counting_evaluation(
                dataset_name=dataset_name, image_dir=image_dir,
                counts_file=counts_file, output_dir=output_dir,
                use_gpu=use_gpu, diameter=30.)
        elif model == 'cellpose3':
            import eval_cellpose3_counting
            result_data = eval_cellpose3_counting.run_counting_evaluation(
                dataset_name=dataset_name, image_dir=image_dir,
                counts_file=counts_file, output_dir=output_dir,
                use_gpu=use_gpu, diameter=0.)
        elif model == 'microatlas':
            import eval_microatlas_counting
            result_data = eval_microatlas_counting.run_counting_evaluation(
                dataset_name=dataset_name, image_dir=image_dir,
                counts_file=counts_file, output_dir=output_dir,
                use_gpu=use_gpu, diameter=30.)
        elif model == 'microsam':
            import eval_microsam_counting
            result_data = eval_microsam_counting.run_counting_evaluation(
                dataset_name=dataset_name, image_dir=image_dir,
                counts_file=counts_file, output_dir=output_dir,
                use_gpu=use_gpu, model_type='vit_l_lm')
        elif model == 'cellsam':
            import eval_cellsam_counting
            result_data = eval_cellsam_counting.run_counting_evaluation(
                dataset_name=dataset_name, image_dir=image_dir,
                counts_file=counts_file, output_dir=output_dir,
                use_gpu=use_gpu)
        else:
            print(f'  [Error] Unknown model: {model}')
            return False, 0, {}
    except Exception as e:
        import traceback
        print(f'  [Error] {e}')
        traceback.print_exc()
        return False, time.time() - start_t, {}

    elapsed = time.time() - start_t
    metrics = result_data.get('metrics', {}) if result_data else {}
    return True, elapsed, metrics


# Execute all tasks
results = []
for dataset_name in DATASETS:
    for model in MODELS:
        print(f'\n{"="*60}')
        print(f'{dataset_name} - {model}')
        print(f'{"="*60}')
        success, elapsed, metrics = run_single_task(model, dataset_name, OUTPUT_DIR, USE_GPU)
        status = 'OK' if success else 'FAIL'
        elapsed_str = f'{elapsed/60:.1f}min' if elapsed > 120 else f'{elapsed:.0f}s'
        print(f'  [{status}] Elapsed: {elapsed_str}')
        if metrics:
            print(f'  MAE={metrics.get("MAE", "N/A")}, R²={metrics.get("R2", "N/A")}, '
                  f'Pearson r={metrics.get("Pearson_r", "N/A")}')
        results.append({
            'dataset': dataset_name, 'model': model,
            'success': success, 'elapsed': elapsed, 'metrics': metrics,
        })

## Generate Excel Summary

In [ ]:
from batch_counting import save_excel_summary

# Generate summary Excel from results
save_excel_summary(results, OUTPUT_DIR)
print(f'\nSummary saved to: {Path(OUTPUT_DIR) / "results_summary.xlsx"}')

## View Results Summary

In [ ]:
# Display results table
print(f'{"Dataset":<10s} {"Model":<12s} {"MAE":>8s} {"RMSE":>8s} {"R²":>8s} {"Pearson r":>10s} {"MPE(%)":>8s}')
print('-' * 66)
for r in results:
    if r['success'] and r['metrics']:
        m = r['metrics']
        print(f'{r["dataset"]:<10s} {r["model"]:<12s} '
              f'{m.get("MAE", 0):>8.2f} {m.get("RMSE", 0):>8.2f} '
              f'{m.get("R2", 0):>8.4f} {m.get("Pearson_r", 0):>10.4f} '
              f'{m.get("MPE", 0):>8.2f}')